# 🎮 Free Minecraft Server on Kaggle

Automatically set up a Minecraft server using Kaggle's free compute resources.

**Important limitations:**
- Kaggle notebooks run for max ~9 hours per session
- NO permanent storage — world data is lost when session ends
- Internet access may be restricted depending on your Kaggle account settings
- No inbound connections — need a tunnel for friends to join
- RAM/CPU limited by your Kaggle tier (typically 2 cores, 13-16 GB RAM free tier)

In [ ]:
# Cell 1: Environment Diagnostics
import platform
import subprocess
import shutil
import os
import sys

print("=" * 60)
print("MINECRAFT SERVER ENVIRONMENT DIAGNOSTICS")
print("=" * 60)

# OS Info
print(f"\n🖥️  Operating System: {platform.system()} {platform.release()}")
print(f"📱 Architecture: {platform.machine()}")
print(f"🐍 Python Version: {platform.python_version()}")

# CPU Cores
try:
    cpu_count = os.cpu_count()
    print(f"⚡ CPU Cores: {cpu_count or 'Unknown'}")
except:
    print("⚠️  Could not detect CPU count")

# RAM (in GB)
try:
    with open('/proc/meminfo', 'r') as f:
        mem_info = f.read()
    total_kb = int([line for line in mem_info.split('\n') if line.startswith('MemTotal:')][0].split()[1])
    total_gb = total_kb / (1024**2)
    print(f"💾 Total RAM: {total_gb:.1f} GB")
except Exception as e:
    print(f"⚠️  RAM detection failed: {e}")

# Disk Space
try:
    total, used, free = shutil.disk_usage('/')
    print(f"💽 Disk - Total: {total/(1024**3):.1f}GB, Free: {free/(1024**3):.1f}GB")
except Exception as e:
    print(f"⚠️  Disk detection failed: {e}")

# Check Java
print("\n🔍 Checking Java...")
java_result = subprocess.run(['java', '-version'], capture_output=True, text=True)
if java_result.returncode == 0:
    print(f"✅ Java found: {java_result.stderr.split('\n')[0] if java_result.stderr else 'Unknown version'}")
else:
    print("❌ Java NOT found - this is required for Minecraft server")

# Internet Test
print("\n🌐 Testing Internet Connectivity...")
try:
    import urllib.request
    urllib.request.urlopen('https://www.minecraft.net', timeout=10)
    print("✅ Internet access OK")
except Exception as e:
    print(f"❌ Internet access issue: {e}")

print("\n" + "=" * 60)
print("NOTE: Kaggle sessions are TEMPORARY. World data is lost when")
print("the notebook stops. Export your world before closing!")
print("=" * 60)

In [ ]:
# Cell 2: Configuration and Dependency Installation
import os
import subprocess

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# ===== EDIT THESE SETTINGS =====
SERVER_EDITION = "bedrock"  # "bedrock" or "java"
SERVER_VERSION = "1.21.50"  # Bedrock version (latest stable)
MINECRAFT_VERSION = "1.21.50"  # Minecraft client version
EULA_ACCEPTED = True  # Set to True after reading https://minecraft.net/eula
VIEW_DISTANCE = 6  # Lower = less lag, higher = see farther
SIMULATION_DISTANCE = 4  # Mob tick distance
MAX_PLAYERS = 5
DIFFICULTY = "normal"  # peaceful, easy, normal, hard
SEED = ""  # Leave empty for random, or set a specific seed
MOTD = "Kaggle Minecraft Server"
ONLINE_MODE = True  # Set False for offline/cracked clients

# Memory allocation (in MB) - be conservative!
# For Kaggle free tier, use 1024-2048MB max
JAVA_XMS = 1024  # Initial heap
JAVA_XMX = 2048  # Max heap (DO NOT exceed available RAM)

print(f"📋 Server Edition: {SERVER_EDITION}")
print(f"📋 View Distance: {VIEW_DISTANCE}")
print(f"📋 Max Players: {MAX_PLAYERS}")
print(f"📋 Difficulty: {DIFFICULTY}")
print(f"📋 Seed: {SEED or '(random)'}")
print(f"📋 EULA: {'Accepted' if EULA_ACCEPTED else 'NOT Accepted - server will not start'}")

# Create server directory
SERVER_DIR = "/kaggle/working/minecraft_server"
os.makedirs(SERVER_DIR, exist_ok=True)
print(f"\n📁 Server directory: {SERVER_DIR}")

In [ ]:
# Cell 3: Server Download and Verification
import os
import urllib.request
import zipfile
import hashlib

SERVER_DIR = "/kaggle/working/minecraft_server"
SERVER_EDITION = "bedrock"  # Use the value from Cell 2
SERVER_VERSION = "1.21.50"

print("=" * 60)
print("DOWNLOADING MINECRAFT SERVER")
print("=" * 60)

if SERVER_EDITION == "java":
    print("\n📥 Downloading Paper Minecraft Server (Java Edition)...")
    # PaperMC download URL pattern
    paper_url = f"https://api.papermc.io/v2/projects/paper/versions/1.21.4/builds/496/downloads/paper-1.21.4-496.jar"
    server_jar = os.path.join(SERVER_DIR, "paper.jar")
    
    try:
        urllib.request.urlretrieve(paper_url, server_jar)
        print(f"✅ Downloaded Paper server to {server_jar}")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        print("💡 Try: pip install --upgrade urllib3")

elif SERVER_EDITION == "bedrock":
    print("\n📥 Downloading Minecraft Bedrock Dedicated Server...")
    print("⚠️  Bedrock server binaries are Amazon Linux only.")
    print("📌 Tutorial: Use ngrok to tunnel a local Bedrock server instead.")
    print("\nFor Kaggle, we recommend Java Edition as it's officially supported.")
    print("\nIf you want Bedrock via tunnel:")
    print("  1. Run a Bedrock server locally on your machine")
    print(f"  2. Use ngrok to expose port {19132} (UDP)")
    print("  3. Share the ngrok address with friends")

print("\n" + "=" * 60)

In [ ]:
# Cell 4: Server Configuration
import os

SERVER_DIR = "/kaggle/working/minecraft_server"
EULA_ACCEPTED = True  # From Cell 2

print("=" * 60)
print("GENERATING SERVER CONFIGURATION")
print("=" * 60)

# EULA file
eula_path = os.path.join(SERVER_DIR, "eula.txt")
if EULA_ACCEPTED:
    with open(eula_path, 'w') as f:
        f.write("#By changing the setting below to TRUE you are indicating your agreement to our EULA.\n")
        f.write("#You must reboot the server after changing this option.\n")
        f.write("eula=true\n")
    print("✅ EULA accepted (eula=true)")
    print("   Remember: Run 'stop' in-game to shutdown gracefully")
else:
    print("⚠️  EULA NOT accepted. Server will not start!")
    print("   Go to https://minecraft.net/eula and agree, then set EULA_ACCEPTED=True")

# server.properties for Java Edition
props_path = os.path.join(SERVER_DIR, "server.properties")
with open(props_path, 'w') as f:
    f.write("""#Minecraft server properties
server-name=Kaggle Minecraft Server
gamemode=survival
difficulty=normal
level-type=default
enable-command-block=false
enable-query=false
op-permission-level=4
allow-nether=true
difficulty=normal
hardcore=false
enable-command-block=false
max-players=5
network-compression-threshold=256
online-mode=true
player-idle-timeout=0
max-world-size=29999984
spawn-npcs=true
spawn-animals=true
spawn-monsters=true
spawn-protection=16
view-distance=6
simulation-distance=4
entity-broadcast-range-percentage=100
seed=
motd=Kaggle Minecraft Server
debug=false
prevent-proxy-connections=false
generate-structures=true
spawn-protection=16
spawn-complexity=10
max-entities-cramming=100
max-weapon-damage=256
max-tick-time=60000
max-build-height=256
network-compression-threshold=256
server-ip=
server-port=25565
snooper-enabled=true
white-list=false
max-world-size=29999984
use-native-transport=true
verbose=true
enable-rcon=false
rcon.password=
rcon.port=25575
""")
print("✅ server.properties created")

print("\n📋 Configuration Summary:")
print("  - Gamemode: survival")
  - Difficulty: normal")
  - Max Players: 5")
  - View Distance: 6")
  - Online Mode: true")

print("\n" + "=" * 60)

In [ ]:
# Cell 5: Server Startup
import os
import subprocess
import time

SERVER_DIR = "/kaggle/working/minecraft_server"
JAVA_XMS = 1024
JAVA_XMX = 2048

print("=" * 60)
print("STARTING MINECRAFT SERVER")
print("=" * 60)

server_jar = os.path.join(SERVER_DIR, "paper.jar")

if not os.path.exists(server_jar):
    print("❌ Server jar not found! Run Cell 3 first.")
else:
    print(f"📦 Starting server from: {server_jar}")
    print(f"💾 Memory: {JAVA_XMS}M initial, {JAVA_XMX}M max")
    print("\n⏳ Waiting for server to start...")
    print("   (This may take 1-3 minutes for world generation)\n")
    
    # Start server
    cmd = f'java -Xms{JAVA_XMS}m -Xmx{JAVA_XMX}m -jar {server_jar} nogui'
    
    try:
        server_process = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=SERVER_DIR
        )
        
        print(f"✅ Server process started (PID: {server_process.pid})")
        print("\n📋 To check server status, run the health check cell below")
        print("📋 To stop the server, press the stop button in the notebook toolbar")
        
    except Exception as e:
        print(f"❌ Failed to start server: {e}")

print("\n" + "=" * 60)

In [ ]:
# Cell 6: Public Tunnel Setup
print("=" * 60)
print("TUNNEL SETUP FOR PUBLIC ACCESS")
print("=" * 60)

print("""
⚠️  KAGGLE NETWORKING LIMITATIONS

Kaggle notebooks DO NOT have public IP addresses and cannot accept
inbound connections. To let friends join your server, you need a tunnel.

RECOMMENDED APPROACH: Use ngrok

1. Sign up at https://ngrok.com (free tier available)
2. Get your auth token from https://dashboard.ngrok.com/get-started/your-authtoken
3. Install ngrok: !pip install pyngrok
4. Connect: !ngrok config add-authtoken YOUR_TOKEN
5. Start tunnel: !ngrok tcp 25565

ALTERNATIVE: Cloudflare Tunnel
  - More complex setup but no auth token needed
  - Requires Cloudflare account

IMPORTANT: The tunnel URL/port will CHANGE every session.
Tell your friends the new address each time you start the notebook.
""")

# Install pyngrok (run once)
!pip install pyngrok -q

from pyngrok import ngrok

print("\n🔧 Setting up ngrok tunnel...")

# You need to set this up once at https://dashboard.ngrok.com
# Then paste your token below
NGROK_AUTH_TOKEN = ""  # ← Paste your token here

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    
    try:
        # Start TCP tunnel on port 25565 (Minecraft Java default)
        tunnel = ngrok.connect(25565, "tcp")
        print(f"✅ Tunnel established!")
        print(f"📍 Public address: {tunnel.public_url}")
        print(f"📍 Connection: {tunnel.public_url.replace('tcp://', '')}:25565")
    except Exception as e:
        print(f"❌ Tunnel failed: {e}")
        print("💡 Make sure your auth token is correct")
else:
    print("⚠️  No ngrok auth token provided")
    print("   Get one at: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   Then paste it in the NGROK_AUTH_TOKEN variable above")

print("\n" + "=" * 60)

In [ ]:
# Cell 7: Connection Information and Health Checks
import os
import time

SERVER_DIR = "/kaggle/working/minecraft_server"

print("=" * 60)
print("SERVER STATUS & CONNECTION INFO")
print("=" * 60)

# Check if server is running
server_properties = os.path.join(SERVER_DIR, "server.properties")

print("\n📋 Server Configuration:")
print(f"   Directory: {SERVER_DIR}")
print(f"   Port: 25565 (Java Edition default)")

if os.path.exists(server_properties):
    with open(server_properties, 'r') as f:
        props = f.read()
    
    # Parse key settings
    for line in props.split('\n'):
        if '=' in line and not line.startswith('#'):
            key, val = line.split('=', 1)
            if key.strip() in ['max-players', 'view-distance', 'difficulty', 'online-mode', 'motd']:
                print(f"   {key.strip()}: {val.strip()}")

print("\n📋 How Friends Connect (Java Edition):")
print("   1. Open Minecraft → Multiplayer → Add Server")
print("   2. Server Address: (ngrok address from Cell 6)")
print("   3. Server Port: 25565")
print("   4. OR use the direct connection link")

print("\n📋 Server Logs (last 20 lines):")
# Show recent log entries
log_file = os.path.join(SERVER_DIR, "logs", "latest.log")
if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        lines = f.readlines()
    for line in lines[-20:]:
        print(f"   {line.rstrip()}")
else:
    print("   (No logs yet - server may still be starting)")

print("\n" + "=" * 60)

In [ ]:
# Cell 8: Logs, Troubleshooting, and Shutdown
import os
import shutil
import zipfile
from datetime import datetime

SERVER_DIR = "/kaggle/working/minecraft_server"
BACKUP_DIR = "/kaggle/working/backups"
os.makedirs(BACKUP_DIR, exist_ok=True)

print("=" * 60)
print("LOGS, BACKUP & SHUTDOWN HANDLING")
print("=" * 60)

# ===== BACKUP WORLD =====
print("\n💾 BACKUP OPTIONS:")
print("   Kaggle storage is TEMPORARY. Back up your world before closing!")

world_dir = os.path.join(SERVER_DIR, "world")
if os.path.exists(world_dir):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"world_backup_{timestamp}.zip"
    backup_path = os.path.join(BACKUP_DIR, backup_name)
    
    # Create backup
    with zipfile.ZipFile(backup_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(world_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, world_dir)
                zf.write(file_path, arcname)
    
    print(f"   ✅ Backup created: {backup_name}")
    print(f"   📁 Size: {os.path.getsize(backup_path) / (1024**2):.1f} MB")
    print("   📥 Download from Kaggle: Files → world_backup_*.zip → Download")
else:
    print("   ⚠️  No world folder found (server may not have started yet)")

# ===== RESTORATION INSTRUCTIONS =====
print("\n🔄 RESTORING A BACKUP:")
print("   1. Upload your backup .zip to Kaggle Files")
   2. Extract it: !unzip world_backup_*.zip -d world/")
   3. Start the server normally")

# ===== COMMON ISSUES =====
print("\n🛠️  TROUBLESHOOTING:")
print("""
❌ Server won't start:
   - Check Java is installed: !apt-get update && !apt-get install -y default-jdk
   - Verify EULA was accepted (eula.txt should contain 'eula=true')
   - Check available RAM - reduce JAVA_XMX if OOM errors

❌ Can't connect from outside:
   - Make sure ngrok tunnel is running (Cell 6)
   - Check the public URL is correct
   - Firewall may block - try different tunnel region

❌ Lag/Performance issues:
   - Reduce VIEW_DISTANCE to 4-6
   - Reduce MAX_PLAYERS
   - Use Paper server (already configured) for better performance

❌ World not saving:
   - Always run 'stop' command in-game or send SIGTERM
   - Don't just kill the process
""")

# ===== GRACEFUL SHUTDOWN =====
print("\n🛑 SHUTDOWN: To stop the server properly:")
print("   - Option 1: Type 'stop' in the Minecraft server console")
print("   - Option 2: Use the stop button in Kaggle notebook toolbar")
print("   - Option 3: Send SIGTERM to the server process")

print("\n⚠️  IMPORTANT: Always backup your world before shutting down!")
print("   Kaggle sessions are temporary - world data is lost when session ends.")

print("\n" + "=" * 60)
print("SETUP COMPLETE!")
print("=" * 60)
print("\n📋 NEXT STEPS:")
print("   1. Run Cell 5 to start the server")
print("   2. Wait for world generation (1-3 min)")
print("   3. Run Cell 6 to set up ngrok tunnel")
print("   4. Share the ngrok address with friends")
print("   5. When done, backup world and shut down gracefully")
print("\n🎮 Enjoy your Kaggle Minecraft server!")
print("=" * 60)